# Waymo Open Dataset Camera-LiDAR VLA

This notebook uses a small Waymo-style manifest to connect camera/LiDAR perception summaries with language explanations and driving actions.

In [ ]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(Path('data/sample_waymo_segments.csv'))
df

In [ ]:
def describe_waymo_scene(row):
    caption = [f"{row.time_of_day} driving in {row.location} during {row.weather} weather"]
    caption.append(f"visibility is about {row.visibility_m} meters")
    caption.append(f"nearest vehicle is {row.nearest_vehicle_m} meters ahead")
    caption.append(f"LiDAR reports {row.lidar_objects} tracked objects")
    if row.pedestrians or row.cyclists:
        caption.append(f"vulnerable road users: {row.pedestrians} pedestrians and {row.cyclists} cyclists")
    caption.append(f"traffic light is {row.traffic_light}")
    return "; ".join(caption) + "."


def waymo_action_policy(row):
    if row.traffic_light == 'red':
        return 'stop', 'red traffic light'
    if row.pedestrians > 0 or row.cyclists > 0:
        return 'yield', 'vulnerable road user nearby'
    if row.visibility_m < 50 or row.nearest_vehicle_m < 10:
        return 'slow_down', 'limited visibility or short following distance'
    if not row.lane_available:
        return 'change_lane', 'current lane is unavailable'
    return 'keep_lane', 'scene is stable'


df['caption'] = df.apply(describe_waymo_scene, axis=1)
df[['predicted_action', 'reason']] = df.apply(lambda row: pd.Series(waymo_action_policy(row)), axis=1)
df[['segment_id', 'caption', 'predicted_action', 'reason', 'expected_action']]

## Using Real Waymo Data

Download Waymo Open Dataset segments, parse the TFRecords, aggregate camera and LiDAR features per frame, and reuse the caption/action functions as a baseline VLA policy.

## Portfolio Expansion: End-to-End Mini Project

The cells below turn the starter notebook into a fuller project workflow: schema checks, feature auditing, risk analysis, evaluation, visualization, export, and a real-dataset adapter plan.

## 1. Dataset Health Check

Before training or evaluating a model, inspect whether the manifest has missing values, useful feature columns, and the expected action labels.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if 'df' not in globals():
    raise RuntimeError('Run the earlier data-loading cells first so df is available.')

health_report = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[col].dtype) for col in df.columns],
    'missing_values': [int(df[col].isna().sum()) for col in df.columns],
    'unique_values': [int(df[col].nunique(dropna=True)) for col in df.columns],
})
health_report

## 2. Feature Audit

This step separates numeric, categorical, and text-like fields. In a real dataset version, this helps decide which fields should become perception features, language context, or evaluation targets.

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = [col for col in df.columns if col not in numeric_cols]

audit = {
    'num_rows': len(df),
    'numeric_columns': numeric_cols,
    'categorical_columns': categorical_cols,
    'has_expected_action': 'expected_action' in df.columns,
    'has_predicted_action': 'predicted_action' in df.columns,
}
audit

## 3. Baseline Risk Feature

If the earlier project-specific cells already created a `risk_score`, this cell keeps it. Otherwise it builds a simple normalized risk proxy from numeric perception fields so every project has a comparable analysis column.

In [ ]:
if 'risk_score' not in df.columns:
    usable_numeric = [col for col in numeric_cols if col not in {'match'}]
    if usable_numeric:
        scaled = df[usable_numeric].copy()
        for col in usable_numeric:
            col_min = scaled[col].min()
            col_max = scaled[col].max()
            if col_max != col_min:
                scaled[col] = (scaled[col] - col_min) / (col_max - col_min)
            else:
                scaled[col] = 0
        df['risk_score'] = scaled.mean(axis=1).round(3)
    else:
        df['risk_score'] = 0.0

if 'risk_level' not in df.columns:
    if df['risk_score'].nunique(dropna=True) <= 1:
        df['risk_level'] = 'low'
    else:
        risk_rank = df['risk_score'].rank(method='first', pct=True)
        df['risk_level'] = pd.cut(
            risk_rank,
            bins=[0, 0.34, 0.67, 1.0],
            labels=['low', 'medium', 'high'],
            include_lowest=True
        )

df[['risk_score', 'risk_level']].head()

## 4. Decision Trace Table

A VLA project should show not just the action, but the reason and context behind the action. This table is the compact portfolio artifact for that trace.

In [ ]:
id_candidates = [col for col in df.columns if col.endswith('_id') or col in {'name', 'frame_id', 'segment_id', 'scenario_id', 'step_id', 'sample_id'}]
id_col = id_candidates[0] if id_candidates else df.columns[0]
trace_cols = [id_col]
for col in ['scene_caption', 'caption', 'scene_summary', 'command', 'risk_score', 'risk_level', 'predicted_action', 'reason', 'expected_action']:
    if col in df.columns and col not in trace_cols:
        trace_cols.append(col)

decision_trace = df[trace_cols].copy()
decision_trace

## 5. Evaluation Summary

This evaluates the rule-based policy against the sample expected labels. For a real project, this section can be replaced with validation metrics from manually labeled actions or model predictions.

In [ ]:
if {'predicted_action', 'expected_action'}.issubset(df.columns):
    df['match'] = df['predicted_action'] == df['expected_action']
    accuracy = float(df['match'].mean())
    print(f'Sample policy accuracy: {accuracy:.0%}')
    evaluation_table = pd.crosstab(df['expected_action'], df['predicted_action'], rownames=['expected'], colnames=['predicted'])
else:
    evaluation_table = pd.DataFrame({'note': ['predicted_action and expected_action columns are required for evaluation']})

evaluation_table

## 6. Visual Risk Review

This chart makes the notebook easier to inspect in GitHub and helps compare scenes at a glance.

In [ ]:
plot_df = df[[id_col, 'risk_score']].copy()
ax = plot_df.plot(kind='bar', x=id_col, y='risk_score', legend=False, figsize=(8, 4), color='#2f6fbb')
ax.set_title('VLA risk score by sample')
ax.set_xlabel('Sample')
ax.set_ylabel('Risk score')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()

## 7. Export Results

The export makes each notebook feel like a real project artifact. Generated outputs are ignored by Git so large or repeated experiment files do not clutter the repo.

In [ ]:
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)
export_path = output_dir / 'vla_decision_trace.csv'
decision_trace.to_csv(export_path, index=False)
export_path

## 8. Real Dataset Upgrade Plan

Use this checklist when replacing the sample manifest with the full dataset.

In [ ]:
upgrade_plan = pd.DataFrame([
    {'step': 1, 'task': 'Download the official dataset and keep it outside Git'},
    {'step': 2, 'task': 'Parse raw labels, sensor metadata, or simulator logs into the sample manifest schema'},
    {'step': 3, 'task': 'Add image/LiDAR/map loading only after the tabular baseline is working'},
    {'step': 4, 'task': 'Replace the rule policy with a trained classifier or VLM-based planner'},
    {'step': 5, 'task': 'Track metrics, failures, and example visualizations in the outputs folder'},
])
upgrade_plan